Execute Recommended Data Quality Rules

This notebook executes the recommended data quality rules generated in the previous notebook.

The previous notebook created a rule catalog called `dq_rule_recommendations`.
This notebook reads those recommendations, applies each rule to the bronze tables, counts failed records, and stores the results in a data quality scorecard.

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

CATALOG_NAME = "workspace"
SCHEMA_NAME = "metadata"

CUSTOMERS_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.customers_bronze"
ORDERS_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.orders_bronze"
PRODUCTS_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.products_bronze"

DQ_RULES_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.dq_rule_recommendations"

DQ_RESULTS_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.dq_rule_results"
DQ_SCORECARD_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.dq_scorecard_summary"

RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

customers_df = spark.table(CUSTOMERS_TABLE)
orders_df = spark.table(ORDERS_TABLE)
products_df = spark.table(PRODUCTS_TABLE)

dq_rules_df = spark.table(DQ_RULES_TABLE)

source_tables = {
    "customers": customers_df,
    "orders": orders_df,
    "products": products_df
}

display(dq_rules_df)

for table_name, df in source_tables.items():
    print(f"{table_name}: {df.count()} rows")

In [0]:
# helper functions

def get_source_dataframe(table_name):           # return datafram for a given table name
    if table_name not in source_tables:
        raise ValueError(f"Unknown table name: {table_name}")
    return source_tables[table_name]

def evaluate_row_level_rule(df, rule_sql):      # rules that apply to all rows
    total_records = df.count()
    failed_records = (df.filter(f"NOT ({rule_sql}) OR ({rule_sql}) IS NULL").count())
    passed_records = total_records - failed_records
    return total_records, passed_records, failed_records

def evaluate_unique_rule(df, col_name):      # whether a column is unique
    total_records = df.count()

    duplicate_values_df = (
        df
        .where(F.col(col_name).isNotNull())
        .groupBy(col_name)
        .count()
        .filter(F.col("count") > 1)
    )

    failed_records = (
        df.join(
            duplicate_values_df.select(col_name),
            on=col_name,
            how="inner"
        )
        .count()
    )

    passed_records = total_records - failed_records
    return total_records, passed_records, failed_records

# evaluate referential integrity rules

def determine_status(failed_records):           # converts failed count to pass/fail
    if failed_records == 0:
        return "PASS"
    return "FAIL"

def get_severity_weight(severity):              # assigns numeric weight to severity
    if severity == "high":
        return 3
    if severity == "medium":
        return 2
    if severity == "low":
        return 1
    return 1

In [0]:
def execute_rule(rule):         # execute a recommended data quality rule
    # not_null, regex_format, valid_range, valid_values, data_not_future, unique, referential_integrity
    # unique and referential integreity require special logic


    table_name = rule["table_name"]
    col_name = rule["col_name"]
    rule_type = rule["rule_type"]
    rule_description = rule["rule_description"]
    rule_sql = rule["rule_sql"]
    severity = rule["severity"]

    df = get_source_dataframe(table_name)

    try:
        # unique
        if rule_type == "unique":
            total_records, passed_records, failed_records = evaluate_unique_rule(df=df, col_name=col_name)

        # referential integrity

        # all others
        else:
            total_records, passed_records, failed_records = evaluate_row_level_rule(df=df, rule_sql=rule_sql)

        failure_rate = (
            round((failed_records / total_records) * 100, 2)
            if total_records > 0
            else 0
        )

        status = determine_status(failed_records)
        execution_error = None

    except Exception as error:
        total_records = df.count()
        passed_records = 0
        failed_records = total_records
        failure_rate = 100.0
        status = "ERROR"
        execution_error = str(error)

    return {
        "run_timestamp": RUN_TIMESTAMP,
        "table_name": table_name,
        "col_name": col_name,
        "rule_type": rule_type,
        "rule_description": rule_description,
        "rule_sql": rule_sql,
        "severity": severity,
        "severity_weight": get_severity_weight(severity),
        "total_records": total_records,
        "passed_records": passed_records,
        "failed_records": failed_records,
        "failure_rate": failure_rate,
        "status": status,
        "execution_error": execution_error if execution_error is not None else ""
    }

In [0]:
# executing all rows
rule_rows = [
    row.asDict()
    for row in dq_rules_df.collect()
]

result_rows = []

for rule in rule_rows:
    result = execute_rule(rule)
    result_rows.append(result)

dq_results_df = spark.createDataFrame(result_rows)

display(dq_results_df)

In [0]:
dq_results_df.write.mode("overwrite").format("delta").saveAsTable(DQ_RESULTS_TABLE)

In [0]:
# view failed rules

failed_rules_df = (
    dq_results_df
    .filter(F.col("status") != "PASS")
    .select(
        "table_name",
        "col_name",
        "rule_type",
        "severity",
        "failed_records",
        "failure_rate",
        "status",
        "rule_description",
        "execution_error"
    )
    .orderBy(
        F.desc("severity_weight"),
        F.desc("failed_records")
    )
)

display(failed_rules_df)

In [0]:
scorecard_df = (
    dq_results_df
    .groupBy("table_name")
    .agg(
        F.count("*").alias("total_rules"),
        F.sum(F.when(F.col("status") == "PASS", 1).otherwise(0)).alias("passed_rules"),
        F.sum(F.when(F.col("status") == "FAIL", 1).otherwise(0)).alias("failed_rules"),
        F.sum(F.when(F.col("status") == "ERROR", 1).otherwise(0)).alias("error_rules"),
        F.sum("failed_records").alias("total_failed_records"),
        F.avg("failure_rate").alias("avg_failure_rate"),
        F.sum(
            F.when(
                F.col("status") != "PASS",
                F.col("severity_weight")
            ).otherwise(0)
        ).alias("weighted_failure_score")
    )
    .withColumn(
        "rule_pass_rate",
        F.round((F.col("passed_rules") / F.col("total_rules")) * 100, 2)
    )
    .withColumn(
        "avg_failure_rate",
        F.round(F.col("avg_failure_rate"), 2)
    )
)

In [0]:
scorecard_df = (
    scorecard_df
    .withColumn(
        "dq_health_status",
        F.when(F.col("error_rules") > 0, "ERROR")
         .when(F.col("weighted_failure_score") >= 10, "HIGH_RISK")
         .when(F.col("weighted_failure_score") >= 5, "MEDIUM_RISK")
         .when(F.col("failed_rules") > 0, "LOW_RISK")
         .otherwise("HEALTHY")
    )
    .orderBy(F.desc("weighted_failure_score"))
)

display(scorecard_df)

In [0]:
scorecard_df.write.mode("overwrite").format("delta").saveAsTable(
    DQ_SCORECARD_TABLE
)

In [0]:
display(
    dq_results_df
    .groupBy("severity", "status")
    .agg(
        F.count("*").alias("rule_count"),
        F.sum("failed_records").alias("failed_records")
    )
    .orderBy("severity", "status")
)

In [0]:
display(
    dq_results_df
    .groupBy("rule_type", "status")
    .agg(
        F.count("*").alias("rule_count"),
        F.sum("failed_records").alias("failed_records")
    )
    .orderBy("rule_type", "status")
)

In [0]:
display(
    dq_results_df
    .filter(F.col("failed_records") > 0)
    .select(
        "table_name",
        "col_name",
        "rule_type",
        "severity",
        "failed_records",
        "failure_rate",
        "rule_description"
    )
    .orderBy(
        F.desc("failed_records"),
        F.desc("failure_rate")
    )
)